In [1]:
import pandas as pd

tourist_df = pd.read_csv("Delhi_Tourist_Places_Final.csv")
hotels_df = pd.read_csv("delhi_hotels_final_processed.csv")
restaurants_df = pd.read_csv("zom.csv")
hospital_df = pd.read_csv("final_hospital.csv")

In [2]:
tourist_df.head()

,Zone,State,City,Name,Type,Establishment Year,time needed to visit in hrs,Google review rating,Entrance Fee in INR,Airport with 50km Radius,Weekly Off,Significance,DSLR Allowed,Number of google review in lakhs,Best Time to visit,Full_Address,Latitude,Longitude
0,Northern,Delhi,Delhi,India Gate,War Memorial,1921.0,0.5,4.6,0,Yes,Not Available,Historical,Yes,2.60,Evening,"India Gate, Delhi, India",28.612933,77.229493
1,Northern,Delhi,Delhi,Humayun's Tomb,Tomb,1572.0,2.0,4.5,30,Yes,Not Available,Historical,Yes,0.40,Afternoon,"Humayun's Tomb, Delhi, India",28.593286,77.250647
2,Northern,Delhi,Delhi,Akshardham Temple,Temple,2005.0,5.0,4.6,60,Yes,Not Available,Religious,No,0.40,Afternoon,"Akshardham Temple, Delhi, India",28.612517,77.277318
3,Northern,Delhi,Delhi,Waste to Wonder Park,Theme Park,2019.0,2.0,4.1,50,Yes,Monday,Environmental,Yes,0.27,Evening,"Waste to Wonder Park, Delhi, India",28.583400,77.257400
4,Northern,Delhi,Delhi,Jantar Mantar,Observatory,1724.0,2.0,4.2,15,Yes,Not Available,Scientific,Yes,0.31,Morning,"Jantar Mantar, Delhi, India",28.627179,77.216617


In [3]:
print("Tourist:", tourist_df.shape)
print("Hotels:", hotels_df.shape)
print("Restaurants:", restaurants_df.shape)
print("Hospitals:", hospital_df.shape)

Tourist: (19, 18)
Hotels: (65, 13)
Restaurants: (1881, 5598)
Hospitals: (320, 4)


In [4]:
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    
    return R * c

In [5]:
selected_place = tourist_df.iloc[0]

selected_place[["Name", "Latitude", "Longitude"]]

Name         India Gate
Latitude      28.612933
Longitude     77.229493
Name: 0, dtype: object

In [6]:
ref_lat = selected_place["Latitude"]
ref_lon = selected_place["Longitude"]

print(ref_lat, ref_lon)

28.6129332 77.2294928


In [7]:
hotels_df.columns

Index(['Hotel Name', 'Rating', 'Rating Description', 'Reviews', 'Star Rating',
       'Location', 'Nearest Landmark', 'Distance to Landmark', 'Price', 'Tax',
       'Full_Address', 'Latitude', 'Longitude'],
      dtype='object')

In [8]:
hotels_df["distance_km"] = hotels_df.apply(
    lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
    axis=1
)

hotels_df[["Hotel Name", "distance_km"]].head()

,Hotel Name,distance_km
0,"Welcomhotel by ITC Hotels, Dwarka, New Delhi",17.071813
1,The Suryaa New Delhi,6.436844
2,The Hosteller Delhi | Private Rooms & Dorms,6.436844
3,The LaLiT New Delhi,2.278598
4,Radisson Blu Plaza Delhi Airport,15.509415


In [9]:
nearest_hotels = hotels_df.sort_values("distance_km").head(5)

nearest_hotels[["Hotel Name", "Rating", "Price", "distance_km"]]

,Hotel Name,Rating,Price,distance_km
40,The Hans,3.6,"4,860",1.524587
36,"The Connaught, New Delhi - IHCL SeleQtions",4.5,"7,499",2.168921
62,Jukaso Inn Down Town,3.6,"3,134",2.168921
15,The Park New Delhi,3.5,"7,905",2.168921
37,New Delhi YMCA Tourist Hostel | Rooms & Wi-Fi\...,3.5,"3,148",2.168921


In [10]:
restaurants_df.columns

Index(['Restaurant_Name', 'Pricing_for_2', 'Dining_Rating',
       'Dining_Review_Count', 'Delivery_Rating', 'Delivery_Rating_Count',
       'Latitude', 'Longitude', 'Known_For2', 'Known_For22',
       ...
       'KF3_ Wraps, Oreo Shake, Sandwiches, Burgers',
       'KF3_ Wraps, Sandwiches, Burgers, Pasta',
       'KF3_ Wraps, Veg Burger, Sandwiches, French Fries',
       'KF3_ Young Crowd, Customizable Food, Food Quality, Worth the Money',
       'KF3_ Young Crowd, Elaborate Menu, Fresh Food, Good Quality',
       'KF3_ Young Crowd, Student Crowd, Customizable Food, Good Wifi',
       'KF3_ and Parsi themed ambiance', 'KF3_ and curries to choose from',
       'KF3_ and traditional practices of undivided India, make for an immersive food experience of a slow life so we can again relive the joys of nutritious, healthy, and gratifying food.',
       'KF3_800...Read More'],
      dtype='object', length=5598)

In [11]:
nearest_restaurants = restaurants_df.sort_values("distance_km").head(5)

nearest_restaurants[["Restaurant_Name", "Dining_Rating", "distance_km"]]

KeyError: 'distance_km'

In [12]:
hospital_df["distance_km"] = hospital_df.apply(
    lambda row: haversine(ref_lat, ref_lon, row["LATITUDE"], row["LONGITUDE"]),
    axis=1
)

nearest_hospitals = hospital_df.sort_values("distance_km").head(5)

nearest_hospitals[["Hospital Name", "distance_km"]]

,Hospital Name,distance_km
209,Lions Kidney Hospital & Urology Research Insitute,2.591005
18,Central Hospital,2.921818
142,Thecity Clinic (Unitof Siddharth Medical Servi...,3.240850
154,Manav Medicare Centre,3.560387
19,Chaudhary Eye Centre & Laser Vision,3.568751


In [13]:
hospital_df.columns

Index(['Hospital Name', 'Address', 'LATITUDE', 'LONGITUDE', 'distance_km'], dtype='object')

In [14]:
tourist_df["distance_km"] = tourist_df.apply(
    lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
    axis=1
)

# Remove the selected tourist place (India Gate)
nearby_places = tourist_df[tourist_df["distance_km"] > 0]

nearby_places = nearby_places.sort_values("distance_km").head(5)

nearby_places[["Name", "distance_km"]]

,Name,distance_km
12,National Gallery of Modern Art,0.586226
8,Agrasen ki Baoli,1.524587
11,Lodhi Garden,2.005523
4,Jantar Mantar,2.022103
13,National Zoological Park,2.067842


In [16]:
# ==============================
# USER INPUT (CASE INSENSITIVE)
# ==============================

place_name = input("Enter Tourist Place Name: ").strip()

# Create lowercase column for safe matching
tourist_df["Name_lower"] = tourist_df["Name"].str.lower()

if place_name.lower() not in tourist_df["Name_lower"].values:
    print("Tourist place not found. Please check spelling.")
else:
    selected_place = tourist_df[tourist_df["Name_lower"] == place_name.lower()].iloc[0]
    
    ref_lat = selected_place["Latitude"]
    ref_lon = selected_place["Longitude"]
    
    print("\n==============================")
    print("Selected Place:", selected_place["Name"])
    print("Latitude:", ref_lat)
    print("Longitude:", ref_lon)
    print("==============================")
    
    
    # ==============================
    # HOTELS
    # ==============================
    hotels_df["distance_km"] = hotels_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
        axis=1
    )
    
    nearest_hotels = hotels_df.sort_values("distance_km").head(5)
    
    print("\n🏨 Top 5 Nearby Hotels:")
    print(nearest_hotels[["Hotel Name", "Rating", "Price", "distance_km"]])
    
    
    # ==============================
    # RESTAURANTS
    # ==============================
    restaurants_df["distance_km"] = restaurants_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
        axis=1
    )
    
    nearest_restaurants = restaurants_df.sort_values("distance_km").head(5)
    
    print("\n🍽️ Top 5 Nearby Restaurants:")
    print(nearest_restaurants[["Restaurant_Name", "Dining_Rating", "distance_km"]])
    
    
    # ==============================
    # HOSPITALS
    # ==============================
    hospital_df["distance_km"] = hospital_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["LATITUDE"], row["LONGITUDE"]),
        axis=1
    )
    
    nearest_hospitals = hospital_df.sort_values("distance_km").head(5)
    
    print("\n🏥 Top 5 Nearby Hospitals:")
    print(nearest_hospitals[["Hospital Name", "distance_km"]])
    
    
    # ==============================
    # NEARBY TOURIST PLACES
    # ==============================
    tourist_df["distance_km"] = tourist_df.apply(
        lambda row: haversine(ref_lat, ref_lon, row["Latitude"], row["Longitude"]),
        axis=1
    )
    
    nearby_places = tourist_df[tourist_df["distance_km"] > 0]
    nearby_places = nearby_places.sort_values("distance_km").head(5)
    
    print("\n🌍 Top 5 Nearby Tourist Places:")
    print(nearby_places[["Name", "distance_km"]])

Enter Tourist Place Name:  india gate



Selected Place: India Gate
Latitude: 28.6129332
Longitude: 77.2294928

🏨 Top 5 Nearby Hotels:
                                           Hotel Name  Rating  Price  \
40                                           The Hans     3.6  4,860   
36         The Connaught, New Delhi - IHCL SeleQtions     4.5  7,499   
62                               Jukaso Inn Down Town     3.6  3,134   
15                                 The Park New Delhi     3.5  7,905   
37  New Delhi YMCA Tourist Hostel | Rooms & Wi-Fi\...     3.5  3,148   

    distance_km  
40     1.524587  
36     2.168921  
62     2.168921  
15     2.168921  
37     2.168921  

🍽️ Top 5 Nearby Restaurants:
    Restaurant_Name  Dining_Rating  distance_km
66     Chor Bizarre       0.455229     0.469586
325         Ichiban       0.158341     0.540432
168        Havemore       0.257304     0.540737
19           Gulati       0.554192     0.551400
768           Pindi      -0.039585     0.554415

🏥 Top 5 Nearby Hospitals:
                   

In [17]:
hotels_df["Price"] = hotels_df["Price"].replace(',', '', regex=True).astype(float)

In [18]:
features = ["Rating", "Price", "Star Rating", "Distance to Landmark"]

X = hotels_df[features]

X.head()

,Rating,Price,Star Rating,Distance to Landmark
0,4.0,6223.0,5.0,NaN
1,4.1,5348.0,5.0,NaN
2,4.3,2228.0,NaN,NaN
3,4.1,9040.0,5.0,NaN
4,4.3,4500.0,5.0,5.2 km


In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [20]:
# Remove commas from Price and convert to float
hotels_df["Price"] = hotels_df["Price"].astype(str).str.replace(",", "")
hotels_df["Price"] = pd.to_numeric(hotels_df["Price"], errors="coerce")

# Remove "km" from Distance to Landmark
hotels_df["Distance to Landmark"] = hotels_df["Distance to Landmark"].astype(str).str.replace(" km", "")
hotels_df["Distance to Landmark"] = pd.to_numeric(hotels_df["Distance to Landmark"], errors="coerce")

# Convert other columns to numeric
hotels_df["Rating"] = pd.to_numeric(hotels_df["Rating"], errors="coerce")
hotels_df["Star Rating"] = pd.to_numeric(hotels_df["Star Rating"], errors="coerce")

hotels_df[["Rating","Price","Star Rating","Distance to Landmark"]].head()

,Rating,Price,Star Rating,Distance to Landmark
0,4.0,6223.0,5.0,NaN
1,4.1,5348.0,5.0,NaN
2,4.3,2228.0,NaN,NaN
3,4.1,9040.0,5.0,NaN
4,4.3,4500.0,5.0,5.2


In [21]:
X = X.fillna(X.mean())

X.head()

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]

ValueError: could not convert string to float: '5.2 km '

In [23]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=6, metric="euclidean")

knn.fit(X_scaled)

NameError: name 'X_scaled' is not defined

In [24]:
def recommend_similar_hotels(hotel_name):

    # Find the index of the selected hotel
    hotel_index = hotels_df[hotels_df["Hotel Name"] == hotel_name].index[0]

    # Get the feature vector
    hotel_vector = X_scaled[hotel_index].reshape(1, -1)

    # Find nearest hotels
    distances, indices = knn.kneighbors(hotel_vector)

    # Remove the first one (it is the same hotel)
    similar_hotels = hotels_df.iloc[indices[0][1:]]

    return similar_hotels[["Hotel Name", "Rating", "Price", "Star Rating"]]

In [25]:
recommend_similar_hotels("The LaLiT New Delhi")

NameError: name 'X_scaled' is not defined

In [26]:
features_rest = ["Dining_Rating", "Pricing_for_2"]

X_rest = restaurants_df[features_rest]

X_rest.head()

,Dining_Rating,Pricing_for_2
0,0.752118,1.080353
1,0.752118,0.117007
2,0.752118,0.545161
3,0.752118,0.438122
4,0.752118,-0.525224


In [27]:
from sklearn.preprocessing import StandardScaler

scaler_rest = StandardScaler()

X_rest_scaled = scaler_rest.fit_transform(X_rest)

X_rest_scaled[:5]

array([[ 3.81244811,  1.70634793],
       [ 3.81244811,  0.36402618],
       [ 3.81244811,  0.96061362],
       [ 3.81244811,  0.81146676],
       [ 3.81244811, -0.53085499]])

In [28]:
from sklearn.neighbors import NearestNeighbors

knn_rest = NearestNeighbors(n_neighbors=6, metric="euclidean")

knn_rest.fit(X_rest_scaled)

print("Restaurant KNN model trained")

Restaurant KNN model trained


In [29]:
def recommend_similar_restaurants(restaurant_name):

    # Find index of selected restaurant
    rest_index = restaurants_df[restaurants_df["Restaurant_Name"] == restaurant_name].index[0]

    # Get feature vector
    rest_vector = X_rest_scaled[rest_index].reshape(1, -1)

    # Find nearest restaurants
    distances, indices = knn_rest.kneighbors(rest_vector)

    # Remove first result (same restaurant)
    similar_restaurants = restaurants_df.iloc[indices[0][1:]]

    return similar_restaurants[["Restaurant_Name", "Dining_Rating", "Pricing_for_2"]]

In [30]:
recommend_similar_restaurants("Gulati")

,Restaurant_Name,Dining_Rating,Pricing_for_2
22,Bo Tai,0.554192,1.508507
28,Downtown - Diners & Living Beer Cafe,0.554192,1.187391
9,Comorin,0.653155,1.508507
13,Pa Pa Ya,0.653155,1.508507
73,The Great Kabab Factory - Radisson Blu Plaza D...,0.455229,1.508507
